In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
cm = np.load("cm.npy")

In [ ]:
sns.heatmap(
    cm / cm.sum(axis=1), cmap="Blues", yticklabels=[], xticklabels=[]
)
plt.show()

In [ ]:
diag = np.diag(cm / cm.sum(axis=1))

In [ ]:
sns.histplot(diag, bins=10, kde=True)
plt.ylabel("Counts")
plt.xlabel("Probability")
plt.show()

# Metrics

In [ ]:
def compute_multiclass_metrics(confusion_matrix):
    # Number of classes
    num_classes = confusion_matrix.shape[0]
    
    # Initialize dictionaries to hold metrics for each class
    metrics_per_class = {}
    
    # Initialize variables for macro and micro averaging
    total_tp = total_fp = total_fn = total_tn = 0
    
    # Compute metrics for each class
    for i in range(num_classes):
        tp = confusion_matrix[i, i]
        fp = confusion_matrix[:, i].sum() - tp
        fn = confusion_matrix[i, :].sum() - tp
        tn = confusion_matrix.sum() - (tp + fp + fn)
        
        # Compute metrics for the current class
        accuracy = (tp + tn) / (tp + fp + fn + tn)
        precision = tp / (tp + fp) if (tp + fp) != 0 else 0
        recall = tp / (tp + fn) if (tp + fn) != 0 else 0
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
        
        # Store metrics for the current class
        metrics_per_class[i] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1_score,
            'Specificity': specificity
        }
        
        # Update totals for micro averaging
        total_tp += tp
        total_fp += fp
        total_fn += fn
        total_tn += tn
    
    # Compute overall metrics (macro average)
    macro_avg_metrics = {
        'Accuracy': np.mean([metrics_per_class[i]['Accuracy'] for i in range(num_classes)]),
        'Precision': np.mean([metrics_per_class[i]['Precision'] for i in range(num_classes)]),
        'Recall': np.mean([metrics_per_class[i]['Recall'] for i in range(num_classes)]),
        'F1 Score': np.mean([metrics_per_class[i]['F1 Score'] for i in range(num_classes)]),
        'Specificity': np.mean([metrics_per_class[i]['Specificity'] for i in range(num_classes)])
    }
    
    # Compute overall metrics (micro average)
    micro_avg_metrics = {
        'Accuracy': (total_tp + total_tn) / (total_tp + total_fp + total_fn + total_tn),
        'Precision': total_tp / (total_tp + total_fp) if (total_tp + total_fp) != 0 else 0,
        'Recall': total_tp / (total_tp + total_fn) if (total_tp + total_fn) != 0 else 0,
        'F1 Score': (2 * total_tp) / (2 * total_tp + total_fp + total_fn) if (2 * total_tp + total_fp + total_fn) != 0 else 0,
        'Specificity': total_tn / (total_tn + total_fp) if (total_tn + total_fp) != 0 else 0
    }
    
    return metrics_per_class, macro_avg_metrics, micro_avg_metrics

In [ ]:
metrics_per_class, macro_avg_metrics, micro_avg_metrics = compute_multiclass_metrics(cm)

In [ ]:
print("Metrics per class:")
for cls, metrics in metrics_per_class.items():
    print(f'Class {cls}:')
    for metric, value in metrics.items():
        print(f'  {metric}: {value:.4f}')

print("\nMacro Average Metrics:")
for metric, value in macro_avg_metrics.items():
    print(f'{metric}: {value:.4f}')

print("\nMicro Average Metrics:")
for metric, value in micro_avg_metrics.items():
    print(f'{metric}: {value:.4f}')